In [ ]:
import sys, pathlib

# Locate the repo's src/ regardless of the kernel's working directory by walking
# up until we find src/bb_code_sim.py.
_cwd = pathlib.Path.cwd().resolve()
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "src" / "bb_code_sim.py").exists():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise RuntimeError(f"could not locate repo src/ starting from cwd={_cwd}")

# Importance Sampling vs. Direct Monte Carlo

This notebook compares the **weight-stratified importance sampling** estimator (`importance_sampling.py`, based on arXiv:2511.15177) against direct Monte Carlo for estimating the logical error rate of a surface code.

### Why importance sampling?
Direct Monte Carlo at small $p$ requires huge shot counts because logical errors are rare — at $p \ll p_\text{threshold}$ you may need millions of shots just to see one.

Importance sampling sidesteps this by:
1. Sampling fault sets at fixed weight $w$ uniformly at random (not at rate $p$)
2. Estimating $f(w) = P(\text{decoder fails} \mid \text{weight } w)$ — these don't go to zero as $p$ shrinks
3. Reweighting analytically: $P_L(p) = \sum_w f(w) \binom{N}{w} q^w (1-q)^{N-w}$

One IS sampling pass gives you the entire LER vs $p$ curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

from surface_code_sim import SurfaceCodeSimulator, ErrorModel, PyMatchingDecoder
from importance_sampling import importance_sample

SEED = 42
DISTANCE = 5
ROUNDS   = 5

sim = SurfaceCodeSimulator(distance=DISTANCE)
print(f'Surface code: d={DISTANCE}, rounds={ROUNDS}')

## 1. Direct Monte Carlo sweep

Sweep $p$ over ~2 decades. Below threshold we need many shots; above threshold a few hundred suffice.

In [ ]:
# {p: shots} — more shots at low p where LER is small
p_shots_mc = {
    0.002: 50_000,
    0.004: 30_000,
    0.007: 15_000,
    0.010: 10_000,
    0.013:  8_000,
    0.017:  6_000,
    0.022:  5_000,
}

mc_results = []
t0 = time.perf_counter()
for p, shots in tqdm(p_shots_mc.items(), desc='direct MC'):
    r = sim.run(ErrorModel.symmetric(p), rounds=ROUNDS, shots=shots, seed=SEED)
    mc_results.append(r)
    print(f'  p={p:.4f}  shots={shots:>6}  LER={r.logical_error_rate:.5f} ± {r.logical_error_rate_se:.5f}  (n_err={r.num_logical_errors})')

print(f'\nTotal direct-MC time: {time.perf_counter()-t0:.1f}s')

## 2. Importance sampling at a single reference $p$

Build the circuit once at a reference rate `p_ref` and sample at fault weights $w = 1, \ldots, 12$. The reweighting formula then gives $P_L(p)$ at **any** target $p$ from a single pass.

In [ ]:
P_REF = 0.01
WEIGHTS = list(range(1, 13))
SHOTS_PER_WEIGHT = 500

circuit = sim.build_circuit(ErrorModel.symmetric(P_REF), rounds=ROUNDS)

p_targets = np.logspace(np.log10(0.002), np.log10(0.025), 30)

t0 = time.perf_counter()
is_result = importance_sample(
    circuit,
    PyMatchingDecoder(),
    p_ref=P_REF,
    p_values=p_targets,
    weights=WEIGHTS,
    shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)
is_elapsed = time.perf_counter() - t0

total_is_shots = len(WEIGHTS) * SHOTS_PER_WEIGHT
print(f'IS total shots: {total_is_shots}  ({len(WEIGHTS)} weights × {SHOTS_PER_WEIGHT} shots each)')
print(f'IS time: {is_elapsed:.1f}s')
print(f'IS gives {len(p_targets)} p points from ONE sampling pass.')
print()
print('Failure spectrum f(w):')
for w, T, F in zip(is_result.spectrum.weights, is_result.spectrum.trials, is_result.spectrum.failures):
    print(f'  w={w:>2}:  F/T = {F:>4}/{T}  =  {F/T:.3f}')
print(f'\nN_expanded = {is_result.spectrum.n_expanded}')
print(f'q_base     = {is_result.spectrum.q_base:.5f}')

## 3. Comparison plot

The IS curve (with its error band) should agree with the direct-MC points wherever MC has reliable statistics. At low $p$ the MC error bars blow up while IS stays tight.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# IS curve with shaded uncertainty
lo = np.maximum(is_result.P_logical - is_result.P_logical_se, 1e-12)
hi = is_result.P_logical + is_result.P_logical_se
ax.fill_between(is_result.p_values, lo, hi, color='steelblue', alpha=0.3, label='IS ± SE')
ax.plot(is_result.p_values, is_result.P_logical, '-', color='steelblue',
        label=f'Importance sampling ({total_is_shots} shots)')

# Direct MC points
p_mc   = np.array(list(p_shots_mc.keys()))
ler_mc = np.array([r.logical_error_rate for r in mc_results])
se_mc  = np.array([r.logical_error_rate_se for r in mc_results])
shots_mc_total = sum(p_shots_mc.values())
ax.errorbar(p_mc, ler_mc, yerr=se_mc, fmt='o', capsize=4, color='tomato', zorder=5,
            label=f'Direct MC ({shots_mc_total:,} shots total)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate')
ax.set_title(f'IS vs direct MC — surface code d={DISTANCE}, rounds={ROUNDS}')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Shot-budget efficiency

Compare the per-point uncertainty of each method, given how many shots were spent.

In [ ]:
# Interpolate IS estimate onto the MC p values
mc_p   = np.array(list(p_shots_mc.keys()))
is_at_mc_p = np.interp(mc_p, is_result.p_values, is_result.P_logical)
is_se_at_mc_p = np.interp(mc_p, is_result.p_values, is_result.P_logical_se)

print(f'{"p":>8} {"MC shots":>10} {"MC LER ± SE":>22} {"IS LER ± SE":>22} {"|Δ|/SE":>8}')
print('-' * 76)
for p, shots, r, is_ler, is_se in zip(mc_p, p_shots_mc.values(), mc_results, is_at_mc_p, is_se_at_mc_p):
    diff = abs(is_ler - r.logical_error_rate)
    combined_se = np.hypot(is_se, r.logical_error_rate_se)
    z = diff / combined_se if combined_se > 0 else 0
    print(f'{p:>8.4f} {shots:>10,} '
          f'{r.logical_error_rate:>9.5f} ± {r.logical_error_rate_se:.5f}  '
          f'{is_ler:>9.5f} ± {is_se:.5f}  {z:>6.1f}σ')

print()
print(f'Direct MC total shots: {shots_mc_total:,}')
print(f'IS total shots:        {total_is_shots:,}')

## 5. The power of IS at low $p$

Push the target $p$ values down to $10^{-4}$ — well below where direct MC could realistically resolve anything from a few thousand shots. The IS estimate (using the same single sampling pass from §2) extrapolates smoothly.

In [ ]:
# Re-evaluate the existing spectrum at much lower p (no new sampling)
p_lowextend = np.logspace(-4, np.log10(0.025), 50)
is_low = importance_sample(
    circuit, PyMatchingDecoder(),
    p_ref=P_REF, p_values=p_lowextend,
    weights=WEIGHTS,
    shots_per_weight=SHOTS_PER_WEIGHT,
    seed=SEED,
)

fig, ax = plt.subplots(figsize=(8, 5))
lo = np.maximum(is_low.P_logical - is_low.P_logical_se, 1e-15)
hi = is_low.P_logical + is_low.P_logical_se
ax.fill_between(is_low.p_values, lo, hi, color='steelblue', alpha=0.3)
ax.plot(is_low.p_values, is_low.P_logical, '-', color='steelblue', label='IS extrapolation')
ax.errorbar(p_mc, ler_mc, yerr=se_mc, fmt='o', capsize=4, color='tomato', zorder=5, label='Direct MC')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate')
ax.set_title(f'IS extrapolation to $p = 10^{{-4}}$  —  surface code d={DISTANCE}')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()